# AIE — pandas & seaborn, before Session 2

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/racousin/data_science_practice/blob/main/website/public/modules/python-ai-engineering/challenges/aie-s0-pandas-seaborn.ipynb)

Every session from 2 onward reads a CSV, reshapes it, and plots it.
This notebook is the subset of `pandas` and `seaborn` those sessions
actually use — nothing more — with the code in front of you rather
than described.

**Run it before Session 2.** It needs no API key and no account: the
data ships with seaborn. It is the same penguins table Session 2's
*The Data* lesson is built on, so you meet it twice.

The written version is the *Reference — pandas & seaborn* lesson in
Session 1. Read either; do this one.

---

## 1. The DataFrame

A `DataFrame` is a dictionary of **typed columns** sharing one index.
Both halves matter: each column has a single dtype (not each cell),
and columns align by index label rather than by position.

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

df = sns.load_dataset("penguins")
df.head()

### The four lines to run on any new table

In this order, every time, before modelling anything.

In [ ]:
print("shape:", df.shape)          # (n, p)
print()
print(df.dtypes)

`dtypes` is the line people skip and the one that bites. pandas infers
types when it reads: one stray `"N/A"` in a numeric column and the
whole column comes back as text, after which arithmetic on it fails —
or silently concatenates.

Here four columns are `float64` and three are text.

In [ ]:
df.isna().sum()

Missing values, per column. `isna()` returns a frame of booleans the
same shape as `df`, so `.sum()` counts down each column.

In [ ]:
df.describe()

**Look at `count`.** It says 342, and the frame has 344 rows —
`describe` drops missing values without saying so, and that gap is
your first evidence they exist.

---

## 2. Selecting

One bracket gives a **Series**, two give a **DataFrame**. Keep track
of which you are holding.

In [ ]:
print(type(df["species"]))
print(type(df[["species", "island"]]))

Rows come out with a boolean mask. Note `&`, not `and` — and the
parentheses, which are required.

In [ ]:
heavy = df[df["body_mass_g"] > 5000]
print(len(heavy), "penguins over 5kg")

subset = df[(df["sex"] == "Male") & (df["island"] == "Dream")]
print(len(subset), "male penguins on Dream")

`and` genuinely does not work there: `&` compares element by element
over the whole column, while `and` tries to reduce it to one true or
false and raises.

Dropping is by keyword — and the keyword is load-bearing.

In [ ]:
X = df.drop(columns=["species"])   # NOT df.drop("species"), which drops a ROW
print(list(X.columns))

Splitting numeric from categorical is a step every challenge notebook
performs, because the two halves get different treatment.

In [ ]:
print("numeric    :", list(df.select_dtypes("number").columns))
print("categorical:", list(df.select_dtypes(exclude="number").columns))

---

## 3. Summarising — does this column carry signal?

`groupby` answers that before any model does. Read it as **split →
apply → combine**.

In [ ]:
df.groupby("species")["body_mass_g"].agg(["count", "mean", "std"])

The within-species spreads are far below the pooled one, so `species`
explains a large share of the variance in mass. That is the statement
"this feature is informative", made before fitting anything.

### The rate trick

For a **0/1 target the mean of the column is its rate**, which makes
this the single most useful line in exploratory work on a classifier.

In [ ]:
penguins = df.assign(is_gentoo=(df["species"] == "Gentoo").astype(int))
penguins.groupby("island")["is_gentoo"].agg(["count", "mean"])

`count` says which islands are common; `mean` says which are
predictive. Different questions — and it is the second one you want.
Gentoos are only ever found on Biscoe.

`assign` above added a column and returned a new frame, leaving `df`
untouched.

---

## 4. seaborn

Two facts explain most of the API.

**Every plot takes tidy data**: hand it the whole frame and name the
columns. It does the grouping, the aggregation and the legend.

**`hue=` splits any plot by a third column.** It is the highest-value
argument in the library.

In [ ]:
sns.set_theme(style="whitegrid")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.histplot(data=df, x="body_mass_g", ax=axes[0])
sns.histplot(data=df, x="body_mass_g", hue="species", ax=axes[1])
axes[0].set_title("without hue")
axes[1].set_title("with hue='species'")
plt.tight_layout()
plt.show()

The same data. The right-hand panel shows the left-hand one is two
populations, not a wide one.

### Axes-level and figure-level

The distinction behind most "why is my figure blank" questions:

| | draws onto | takes `ax=` |
|---|---|---|
| **axes-level** — `histplot`, `barplot`, `boxplot`, `lineplot`, `regplot`, `heatmap`, `countplot` | a subplot you give it | yes |
| **figure-level** — `pairplot`, `relplot`, `catplot`, `displot` | a whole figure it makes itself | **no** |

So `pairplot` cannot go inside a grid you built. It *is* the grid.

### The plots this course uses

Four axes-level plots, placed into one figure.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
sns.boxplot(data=df, x="species", y="body_mass_g", ax=axes[0, 0])
axes[0, 0].set_title("boxplot — does y differ across levels?")

sns.countplot(data=df, x="island", ax=axes[0, 1])
axes[0, 1].set_title("countplot — how many rows per level?")

sns.regplot(data=df, x="flipper_length_mm", y="body_mass_g",
            scatter_kws=dict(alpha=0.4, s=12), ax=axes[1, 0])
axes[1, 0].set_title("regplot — is the relationship linear?")

sns.heatmap(df.corr(numeric_only=True), annot=True, fmt=".2f",
            cmap="RdBu_r", center=0, ax=axes[1, 1])
axes[1, 1].set_title("heatmap — what is redundant with what?")
plt.tight_layout()
plt.show()

A caution on that last panel. Correlation measures the **linear** part
of a relationship only. A column can drive the target hard and still
show a correlation near zero if its shape is not a line — which is
exactly what happens to `hour` in the Session 2 bike challenge.

### pairplot — the first thing to run on a new table

In [ ]:
sns.pairplot(df, hue="species", height=1.8)
plt.show()

Every numeric pair at once. Read straight off it: flipper length and
body mass are near-linear; Gentoo separates cleanly on almost any
pair; Adelie and Chinstrap overlap everywhere except bill length —
which predicts which two classes a model will confuse, before you
train one.

It costs $p^2$ panels, so on a wide frame restrict it:
`sns.pairplot(df, vars=["bmi", "bp"], hue="target")`.

---

## 5. Into a model, and back out

`scikit-learn` accepts only a rectangle of numbers. One call converts
every non-numeric column to indicator columns and leaves the numeric
ones alone.

In [ ]:
clean = df.dropna()
X = clean.drop(columns=["species"])
y = (clean["species"] == "Gentoo").astype(int)

X_encoded = pd.get_dummies(X)
print(list(X_encoded.columns))

### The alignment trap — the part to actually read

Encode train and test **separately** and they can end up with
different columns: a category in one and not the other changes the
width, or the order. The model is then fed nonsense, usually with no
error at all.

Watch it happen — a training set with no Torgersen penguins:

In [ ]:
train = clean[clean["island"] != "Torgersen"].drop(columns=["species"])
test = clean.drop(columns=["species"])

a = pd.get_dummies(train)
b = pd.get_dummies(test)
print("train columns:", len(a.columns), list(a.columns[-3:]))
print("test  columns:", len(b.columns), list(b.columns[-3:]))
print("same?", list(a.columns) == list(b.columns))

Different widths. Now the fix:

In [ ]:
b_aligned = pd.get_dummies(test).reindex(columns=a.columns, fill_value=0)
print("same?", list(a.columns) == list(b_aligned.columns))
assert list(a.columns) == list(b_aligned.columns)

`reindex(columns=...)` forces the second frame onto the first's exact
column list — missing columns created and filled with `0`, extra
columns dropped. Write the `assert` once and stop worrying about it.

### Fit, predict, write

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score

A, B, a_y, b_y = train_test_split(X_encoded, y, test_size=0.3, random_state=0)
model = LogisticRegression(max_iter=2000).fit(A, a_y)
print("holdout accuracy:", round(model.score(B, b_y), 4))

cv = cross_val_score(LogisticRegression(max_iter=2000), X_encoded, y, cv=5)
print("5-fold CV       :", round(cv.mean(), 4))

**Accuracy 1.0 — and you should be suspicious of that.** Session 3 is
entirely about why a perfect score usually means the model memorised
something rather than learned it.

Here it is real, and the second line is how you know: 5-fold
cross-validation agrees at 0.997, so the result is not an artefact of
one lucky split. Telling Gentoos apart genuinely is easy — go back to
the `pairplot` and look at how far they sit from the other two on
almost every pair. The measurements alone are enough; `island` on its
own only reaches 0.82.

That is the habit: when a score looks too good, do not celebrate and
do not panic — cross-validate, and find out which column earned it.

In [ ]:
out = pd.DataFrame({"id": B.index, "prediction": model.predict(B)})
out.to_csv("submission.csv", index=False)
out.head()

**`index=False` is not cosmetic.** Leave it out and pandas writes its
row labels as an extra unnamed leading column; the scorer sees a
column it did not ask for and rejects the file.

---

## That is the whole surface

Everything the challenge notebooks do is built from what is above. The
habit worth keeping is the four lines from section 1 —
`shape`, `dtypes`, `isna().sum()`, `describe()` — plus one plot of the
target and one `pairplot`, on every dataset, before any modelling.

It costs a minute, and it is the difference between modelling the data
you have and modelling the data you assumed you had.